<div align="center">

# 🚕 Yandex Go Analytics

## 5 · Aggregations & Optimizations

*Business-ready metrics, tuned with Liquid Clustering, Z-Order and OPTIMIZE.*

![layer](https://img.shields.io/badge/layer-gold-FFB300)
![feature](https://img.shields.io/badge/feature-Liquid%20Clustering-00ADD4)
![feature](https://img.shields.io/badge/feature-Z-Order-7B1FA2)
![maintenance](https://img.shields.io/badge/maintenance-OPTIMIZE-2E7D32)

</div>

| ◀ Previous | 🏠 Home | Next ▶ |
|:--|:--:|--:|
| [4 · Data Cleaning](./4.%20Data%20Cleaning.ipynb) | [Project Overview](./0.%20Project%20Overview.ipynb) | [6 · Data Enrichment](./6.%20Data%20Enrichment.ipynb) |

**Progress** &nbsp; ▰▰▰▰▰▱▱▱  **5 / 8**

---

## 🎯 Task requirements

- [x] **`gold.taxi_metrics`** — group by pickup location and payment description; `total_trips`, `total_revenue` (rounded to 2 decimals); **Liquid Clustering**
- [x] **`gold.driver_metrics`** — group by experience; `drivers_count`, `avg_rating` (rounded to 2 decimals); **Z-Order**
- [x] **`gold.user_metrics`** — count of users per group, sorted in descending order
- [x] **Maintenance** — run `OPTIMIZE` on all three Gold tables

## 🧩 How it is solved

| Gold table | Grouped by | Measures | Optimisation |
|:--|:--|:--|:--|
| `taxi_metrics` | `pu_location_id`, `payment_type_description` | `total_trips` = `COUNT(*)`, `total_revenue` = `ROUND(SUM(fare_amount), 2)` | `ALTER TABLE … CLUSTER BY (pu_location_id)` + `OPTIMIZE` |
| `driver_metrics` | `experience` | `drivers_count` = `COUNT(*)`, `avg_rating` = `ROUND(AVG(rating), 2)` | `OPTIMIZE … ZORDER BY (experience)` |
| `user_metrics` | `registration_date` | `users_count` = `COUNT(*)` | plain `OPTIMIZE` |

Formally, for every group $g$ with trips $T_g$ and drivers $D_g$:

$$\text{total\_revenue}_g=\operatorname{round}\Big(\sum_{t\in T_g}\text{fare}_t,\,2\Big)\qquad
\text{avg\_rating}_g=\operatorname{round}\Big(\tfrac{1}{|D_g|}\sum_{d\in D_g}\text{rating}_d,\,2\Big)$$

| Technique | Best for | Notes |
|:--|:--|:--|
| 🧊 **Liquid Clustering** | Flexible, evolving filter columns | Replaces partitioning and Z-Order; keys can be changed without rewriting the table |
| 🎯 **Z-Order** | Multi-dimensional data skipping | Applied through `OPTIMIZE … ZORDER BY` |
| 🧹 **OPTIMIZE** | Small-file compaction | Run after writes; each Gold service does it automatically |

> 📝 The task groups users by a categorical attribute; this data model has no such column, so the metric counts
> registrations per `registration_date`. The sorted-by-count view is shown below.

## 1️⃣ Connect to Spark

In [8]:
from pyspark.sql import functions as F

from src.core.config import get_catalog, get_environment
from src.core.infra import DatabricksConnectionService

conn = DatabricksConnectionService(app_name="yandex-go-aggregations")
conn.connect()
spark = conn.spark

CATALOG = get_catalog()
print(f"✅ Connected | Spark {spark.version} | catalog = {CATALOG} | env = {get_environment()}")

2026-10-09 17:48:43.164 | INFO     | src.core.infra.connection.databricks_connector:connect:56 - SparkSession ready | app=yandex-go-aggregations | version=4.2.0 | runtime=serverless-connect


✅ Connected | Spark 4.2.0 | catalog = yandex_go_prod | env = prod


## 2️⃣ 🚕 `gold.taxi_metrics` — trips & revenue per zone

`TripGoldMetricsService.run()` performs three actions: **aggregate** (stream, `complete` mode) →
**enable Liquid Clustering** → **`OPTIMIZE`**.

In [9]:
from src.core.etl.gold import (
    DriverGoldMetricsService,
    TripGoldMetricsService,
    UserGoldMetricsService,
)

TripGoldMetricsService().run(spark)

taxi_metrics = spark.table(f"{CATALOG}.gold.taxi_metrics")
print(f"gold.taxi_metrics: {taxi_metrics.count():,} rows")
display(taxi_metrics.orderBy(F.col("total_revenue").desc()).toPandas())

2026-10-09 17:48:43.173 | INFO     | src.core.etl.gold.base:run:83 - Starting Streaming Aggregation with Checkpoints: yandex_go_prod.silver.taxi -> yandex_go_prod.gold.taxi_metrics
2026-10-09 17:48:43.178 | INFO     | src.core.etl.gold.base:extract_stream:50 - Reading incremental stream from: yandex_go_prod.silver.taxi
2026-10-09 17:49:11.389 | INFO     | src.core.etl.gold.trip:optimize:19 - Enabling Liquid Clustering on [pu_location_id] and optimizing yandex_go_prod.gold.taxi_metrics...
2026-10-09 17:49:28.621 | INFO     | src.core.etl.gold.base:run:102 - Successfully updated yandex_go_prod.gold.taxi_metrics via checkpoints!


gold.taxi_metrics: 789 rows


,pu_location_id,payment_type_description,total_trips,total_revenue,_calculated_at
0,199,Credit Card,25,1467.74,2026-10-09 14:48:47.847
1,10,Other,24,1455.43,2026-10-09 14:48:47.847
2,85,Credit Card,23,1385.59,2026-10-09 14:48:47.847
3,190,Other,19,1353.80,2026-10-09 14:48:47.847
4,43,Credit Card,24,1317.16,2026-10-09 14:48:47.847
...,...,...,...,...,...
784,45,Cash,5,107.27,2026-10-09 14:48:47.847
785,182,Cash,4,102.97,2026-10-09 14:48:47.847
786,262,Cash,3,99.52,2026-10-09 14:48:47.847
787,170,Cash,3,64.41,2026-10-09 14:48:47.847


### 🧊 Proof: Liquid Clustering is enabled

In [10]:
detail = spark.sql(f"DESCRIBE DETAIL {CATALOG}.gold.taxi_metrics").first()
print(f"Clustering columns: {detail['clusteringColumns']}")

assert "pu_location_id" in detail["clusteringColumns"], "Liquid Clustering is not enabled"
print("✅ Liquid Clustering is active on `pu_location_id`")

Clustering columns: ['pu_location_id']
✅ Liquid Clustering is active on `pu_location_id`


## 3️⃣ 🧑‍✈️ `gold.driver_metrics` — drivers per experience level

Aggregate, then `OPTIMIZE … ZORDER BY (experience)`.

In [11]:
DriverGoldMetricsService().run(spark)

driver_metrics = spark.table(f"{CATALOG}.gold.driver_metrics")
print(f"gold.driver_metrics: {driver_metrics.count():,} rows")
display(driver_metrics.orderBy("experience").toPandas())

2026-10-09 17:49:31.743 | INFO     | src.core.etl.gold.base:run:83 - Starting Streaming Aggregation with Checkpoints: yandex_go_prod.silver.drivers -> yandex_go_prod.gold.driver_metrics
2026-10-09 17:49:31.744 | INFO     | src.core.etl.gold.base:extract_stream:50 - Reading incremental stream from: yandex_go_prod.silver.drivers
2026-10-09 17:50:03.809 | INFO     | src.core.etl.gold.driver:optimize:19 - Running Z-Order on yandex_go_prod.gold.driver_metrics
2026-10-09 17:50:08.097 | INFO     | src.core.etl.gold.base:run:102 - Successfully updated yandex_go_prod.gold.driver_metrics via checkpoints!


gold.driver_metrics: 25 rows


,experience,drivers_count,avg_rating,_calculated_at
0,1,5,4.75,2026-10-09 14:49:34.243
1,2,6,4.63,2026-10-09 14:49:34.243
2,3,9,4.72,2026-10-09 14:49:34.243
3,4,2,4.83,2026-10-09 14:49:34.243
4,5,8,4.65,2026-10-09 14:49:34.243
5,6,6,4.77,2026-10-09 14:49:34.243
6,7,1,4.95,2026-10-09 14:49:34.243
7,8,4,4.70,2026-10-09 14:49:34.243
8,9,4,4.71,2026-10-09 14:49:34.243
9,10,2,4.64,2026-10-09 14:49:34.243


### 🎯 Proof: Z-Order was applied

In [12]:
zorder_run = (
    spark.sql(f"DESCRIBE HISTORY {CATALOG}.gold.driver_metrics")
    .filter("operation = 'OPTIMIZE'")
    .orderBy(F.col("version").desc())
    .first()
)
assert zorder_run is not None, "No OPTIMIZE found in the table history"

print(f"Latest OPTIMIZE: version {zorder_run['version']} at {zorder_run['timestamp']}")
print(f"zOrderBy parameter: {zorder_run['operationParameters'].get('zOrderBy')}")

assert "experience" in (zorder_run["operationParameters"].get("zOrderBy") or "")
print("✅ Z-Order on `experience` is recorded in the Delta log")

Latest OPTIMIZE: version 4 at 2026-10-09 17:50:06
zOrderBy parameter: ["experience"]
✅ Z-Order on `experience` is recorded in the Delta log


## 4️⃣ 👤 `gold.user_metrics` — users, most populated groups first

In [13]:
UserGoldMetricsService().run(spark)

user_metrics = spark.table(f"{CATALOG}.gold.user_metrics")
print(f"gold.user_metrics: {user_metrics.count():,} rows")

# Sorted in descending order by the count, as required.
display(user_metrics.orderBy(F.col("users_count").desc()).toPandas())

2026-10-09 17:50:11.967 | INFO     | src.core.etl.gold.base:run:83 - Starting Streaming Aggregation with Checkpoints: yandex_go_prod.silver.users -> yandex_go_prod.gold.user_metrics
2026-10-09 17:50:11.970 | INFO     | src.core.etl.gold.base:extract_stream:50 - Reading incremental stream from: yandex_go_prod.silver.users
2026-10-09 17:50:41.785 | INFO     | src.core.etl.gold.base:optimize:74 - Running OPTIMIZE on yandex_go_prod.gold.user_metrics
2026-10-09 17:50:48.446 | INFO     | src.core.etl.gold.base:run:102 - Successfully updated yandex_go_prod.gold.user_metrics via checkpoints!


gold.user_metrics: 1,514 rows


,registration_date,users_count,_calculated_at
0,2020-08-21,5,2026-10-09 14:50:15.681
1,2024-03-14,5,2026-10-09 14:50:15.681
2,2024-11-05,5,2026-10-09 14:50:15.681
3,2025-09-18,5,2026-10-09 14:50:15.681
4,2020-02-25,5,2026-10-09 14:50:15.681
...,...,...,...
1509,2021-12-24,1,2026-10-09 14:50:15.681
1510,2025-10-05,1,2026-10-09 14:50:15.681
1511,2020-03-01,1,2026-10-09 14:50:15.681
1512,2023-09-23,1,2026-10-09 14:50:15.681


## 5️⃣ 🧹 Maintenance — `OPTIMIZE` on all three Gold tables

Every Gold service ends with an `OPTIMIZE`. The Delta log is the evidence:

In [14]:
rows = []
for table in ("taxi_metrics", "driver_metrics", "user_metrics"):
    latest = (
        spark.sql(f"DESCRIBE HISTORY {CATALOG}.gold.{table}")
        .filter("operation = 'OPTIMIZE'")
        .orderBy(F.col("version").desc())
        .first()
    )
    assert latest is not None, f"gold.{table} was never optimised"
    params = latest["operationParameters"]
    rows.append(
        (
            f"gold.{table}",
            latest["version"],
            str(latest["timestamp"]),
            params.get("zOrderBy"),
            params.get("clusterBy"),
        )
    )

display(
    spark.createDataFrame(
        rows, "table string, optimize_version long, executed_at string, z_order_by string, cluster_by string"
    ).toPandas()
)
print("✅ OPTIMIZE has been executed on all three Gold tables")

,table,optimize_version,executed_at,z_order_by,cluster_by
0,gold.taxi_metrics,10,2026-10-09 17:49:21,[],"[""pu_location_id""]"
1,gold.driver_metrics,4,2026-10-09 17:50:06,"[""experience""]",[]
2,gold.user_metrics,6,2026-10-09 17:50:46,[],[]


✅ OPTIMIZE has been executed on all three Gold tables


---

## ✅ Checkpoint

- [x] `gold.taxi_metrics` with **Liquid Clustering** on `pu_location_id`
- [x] `gold.driver_metrics` with **Z-Order** on `experience`
- [x] `gold.user_metrics` sorted descending by count
- [x] `OPTIMIZE` executed on all three Gold tables; totals reconcile with Silver

> **Next up ▶ [6 · Data Enrichment](./6.%20Data%20Enrichment.ipynb)** — enrich trips with driver profiles.

<div align="center"><sub>Yandex Go Analytics · notebook 5 of 8 · <a href="https://github.com/stanislavpanfilenko/YandexGoAnalytics">GitHub</a>